In [9]:
from pathlib import Path
import re
import pandas as pd

from utils.helpers import load_dataset

In [5]:
# loading the dataset
path = Path(r"data\implicature\eval\eval_circa_indirect_answer_binary_550.json")
data = load_dataset(path)
data[0]

{'id': 'circa-indirect_answer-5',
 'creation_method': 'translation',
 'phenomenon': 'implicature',
 'category': 'indirect_answer_binary',
 'context': 'X chce vědět, jaké knížky Y rád čte.',
 'utterance': '- A: Čteš rád motivační knížky?\n- B: Zrovna jejich fanoušek nejsem.',
 'question': 'Co mluvčí B svou odpovědí v daném kontextu nepřímo sděluje?',
 'options': [{'label': 'A',
   'type': 'direct_answer_positive',
   'text': 'Ano, čtu rád motivační knížky.'},
  {'label': 'B',
   'type': 'direct_answer_negative',
   'text': 'Ne, nečtu rád motivační knížky.'},
  {'label': 'C',
   'type': 'distractor_lexical',
   'text': 'Zrovna jejich názvy si hledám v online katalogu.'},
  {'label': 'D',
   'type': 'distractor_associative',
   'text': 'Motivační literatura se často zaměřuje na kariérní růst.'}],
 'correct_option': 'B',
 'metadata': {'corpus': 'CIRCA', 'topic': 'books'}}

# I. Preprocessing with UDPipe

In [7]:
# text normalization

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


sample = '  ahoj,\n\n   toto   je   test.\n\n  '
print("original:", repr(sample))
print("normalized:", repr(normalize_text(sample)))

ORIGINAL: '  ahoj,\n\n   toto   je   test.\n\n  '
NORMALIZED: 'ahoj, toto je test.'


In [28]:
from ufal.udpipe import Model
from ufal.udpipe import Pipeline, ProcessingError

MODEL_PATH = r"udpipe\czech-fictree-ud-2.5-191206.udpipe"


In [31]:
# initialize the UDPipe model
model = Model.load(MODEL_PATH)
print("[1/2]model initialized" if model is not None else "model NOT initialized")

#building pipeline
pipeline = Pipeline(
    model,
    "tokenize",          # input is raw text
    Pipeline.DEFAULT,    # tagging
    Pipeline.DEFAULT,    # parsing
    "conllu"             # output format
)
print("[2/2]pipeline built")

[1/2]model initialized
[2/2]pipeline built


In [37]:
# converting conllu
def conllu_to_tokens(conllu_text: str) -> list[dict]:
    """ converts ConLLU format into usable python dictionary (only a subset of attributes is used)"""
    tokens = []

    for line in conllu_text.splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        cols = line.split("\t")
        if len(cols) != 10:
            continue

        token_id = cols[0]

        # skip multiword tokens like 1-2 and empty nodes like 3.1
        if "-" in token_id or "." in token_id:
            continue

        tokens.append({
            "form": cols[1],
            "lemma": cols[2],
            "upos": cols[3],
        })

    return tokens

# wrapping into one function
def parse_text_to_tokens(text: str) -> list[dict]:
    """wrapper function to process text into tokens"""
    if not text:
        return []

    error = ProcessingError()
    conllu = pipeline.process(text, error)

    if error.occurred():
        raise RuntimeError(error.message)

    return conllu_to_tokens(conllu)

In [33]:
# test example
text = "Mluvčí: Kdyby tehdy přišel včas, všechno mohlo dopadnout jinak."
tokens = parse_text_to_tokens(text)

print("number of tokens:", len(tokens))
for tok in tokens:
    print(tok)

number of tokens: 13
{'form': 'Mluvčí', 'lemma': 'Mluvčit', 'upos': 'VERB'}
{'form': ':', 'lemma': ':', 'upos': 'PUNCT'}
{'form': 'Když', 'lemma': 'když', 'upos': 'SCONJ'}
{'form': 'by', 'lemma': 'být', 'upos': 'AUX'}
{'form': 'tehdy', 'lemma': 'tehdy', 'upos': 'ADV'}
{'form': 'přišel', 'lemma': 'přijít', 'upos': 'VERB'}
{'form': 'včas', 'lemma': 'včas', 'upos': 'ADV'}
{'form': ',', 'lemma': ',', 'upos': 'PUNCT'}
{'form': 'všechno', 'lemma': 'všechen', 'upos': 'DET'}
{'form': 'mohlo', 'lemma': 'moci', 'upos': 'VERB'}
{'form': 'dopadnout', 'lemma': 'dopadnout', 'upos': 'VERB'}
{'form': 'jinak', 'lemma': 'jinak', 'upos': 'ADV'}
{'form': '.', 'lemma': '.', 'upos': 'PUNCT'}


# II. Measuting Average Token Length

In [36]:
# count tokens
def count_parsed_tokens(tokens: list[dict], exclude_punct: bool = True) -> int:
    """count tokens from an already parsed token list."""
    if exclude_punct:
        return sum(1 for tok in tokens if tok["upos"] != "PUNCT")
    return len(tokens)

# another example
text = "Mluvčí: Kdyby tehdy přišel včas, všechno mohlo dopadnout jinak."
tokens = parse_text_to_tokens(text)

print(tokens)
print("Including punctuation:", count_parsed_tokens(tokens, exclude_punct=False))
print("Excluding punctuation:", count_parsed_tokens(tokens, exclude_punct=True))

[{'form': 'Mluvčí', 'lemma': 'Mluvčit', 'upos': 'VERB'}, {'form': ':', 'lemma': ':', 'upos': 'PUNCT'}, {'form': 'Když', 'lemma': 'když', 'upos': 'SCONJ'}, {'form': 'by', 'lemma': 'být', 'upos': 'AUX'}, {'form': 'tehdy', 'lemma': 'tehdy', 'upos': 'ADV'}, {'form': 'přišel', 'lemma': 'přijít', 'upos': 'VERB'}, {'form': 'včas', 'lemma': 'včas', 'upos': 'ADV'}, {'form': ',', 'lemma': ',', 'upos': 'PUNCT'}, {'form': 'všechno', 'lemma': 'všechen', 'upos': 'DET'}, {'form': 'mohlo', 'lemma': 'moci', 'upos': 'VERB'}, {'form': 'dopadnout', 'lemma': 'dopadnout', 'upos': 'VERB'}, {'form': 'jinak', 'lemma': 'jinak', 'upos': 'ADV'}, {'form': '.', 'lemma': '.', 'upos': 'PUNCT'}]
Including punctuation: 13
Excluding punctuation: 10


In [ ]:
# another wrapper
def count_tokens_in_text(text: str, exclude_punct: bool = True) -> int:
    """parse text and count its tokens."""
    tokens = parse_text_to_tokens(text)
    return count_parsed_tokens(tokens, exclude_punct=exclude_punct)

In [44]:
def compute_item_length_metrics(item: dict, exclude_punct: bool = True) -> dict:
    """
    compute token-length metrics for a single dataset item 

    uses fields:
    - context
    - utterance
    - options
    
    """

    # main text fields
    context = item.get("context", "") or ""
    utterance = item.get("utterance", "") or ""
    options = item.get("options", []) or []

    # parse once
    context_tokens = parse_text_to_tokens(context)
    utterance_tokens = parse_text_to_tokens(utterance)
    option_token_lists = [
        parse_text_to_tokens(opt.get("text", "") or "")
        for opt in options
    ]

    # count
    context_length = count_parsed_tokens(context_tokens, exclude_punct=exclude_punct)
    utterance_length = count_parsed_tokens(utterance_tokens, exclude_punct=exclude_punct)

    option_lengths = [
        count_parsed_tokens(tokens, exclude_punct=exclude_punct)
        for tokens in option_token_lists
    ]

    options_total_length = sum(option_lengths)
    options_mean_length = options_total_length / len(option_lengths) if option_lengths else 0

    item_total_length = (
        context_length
        + utterance_length
        + options_total_length
    )

    metadata = item.get("metadata", {}) or {}

    return {
        "id": item.get("id", ""),
        "phenomenon": item.get("phenomenon", ""),
        "category": item.get("category", ""),
        "context_length": context_length,
        "utterance_length": utterance_length,
        "options_total_length": options_total_length,
        "options_mean_length": options_mean_length,
        "item_total_length": item_total_length,
    }

In [45]:
# information structure
items = load_dataset(r"data\information-structure\eval\inf_structure_eval_720_v1.json")
print("loaded items:", len(items))

one_result = compute_item_length_metrics(items[0])
one_result

loaded items: 720


{'id': 'information-structure-baseline-1-focus_object',
 'phenomenon': 'information_structure',
 'category': 'baseline_focus_object',
 'context_length': 17,
 'utterance_length': 5,
 'options_total_length': 18,
 'options_mean_length': 4.5,
 'item_total_length': 40}

In [46]:
# presupposition
items = load_dataset(r"data\presupposition\eval\presupposition-accusative-60.json")
print("loaded items:", len(items))

one_result = compute_item_length_metrics(items[0])
one_result

loaded items: 60


{'id': 'presupposition-accusative-1',
 'phenomenon': 'presupposition',
 'category': 'accusative',
 'context_length': 11,
 'utterance_length': 14,
 'options_total_length': 31,
 'options_mean_length': 7.75,
 'item_total_length': 56}

In [47]:
# implicature
items = load_dataset(r"data\implicature\eval\eval_circa_indirect_answer_binary_550.json")
print("loaded items:", len(items))

one_result = compute_item_length_metrics(items[0])
one_result

loaded items: 550


{'id': 'circa-indirect_answer-5',
 'phenomenon': 'implicature',
 'category': 'indirect_answer_binary',
 'context_length': 8,
 'utterance_length': 10,
 'options_total_length': 26,
 'options_mean_length': 6.5,
 'item_total_length': 44}

In [48]:
def compute_dataset_length_metrics(items: list[dict], exclude_punct: bool = True) -> pd.DataFrame:
    """compute item-level length metrics for the whole dataset"""
    rows = [
        compute_item_length_metrics(item, exclude_punct=exclude_punct)
        for item in items
    ]
    return pd.DataFrame(rows)


def aggregate_length_metrics(df: pd.DataFrame, group_by: str | list[str]) -> pd.DataFrame:
    """aggregate length metrics by selected grouping columns"""
    if isinstance(group_by, str):
        group_by = [group_by]

    metric_cols = [
        "context_length",
        "utterance_length",
        "options_total_length",
        "options_mean_length",
        "item_total_length",
    ]

    return (
        df.groupby(group_by)[metric_cols]
        .agg(["mean", "median", "std", "min", "max"])
        .round(2)
    )


def summarize_length_metrics(df: pd.DataFrame) -> pd.Series:
    """return simple global mean values for the whole dataset."""
    metric_cols = [
        "context_length",
        "utterance_length",
        "options_total_length",
        "options_mean_length",
        "item_total_length",
    ]
    return df[metric_cols].mean().round(2)

In [56]:
items = load_dataset(r"data\information-structure\eval\inf_structure_eval_720_v1.json")

df_lengths = compute_dataset_length_metrics(items)

global_summary = summarize_length_metrics(df_lengths)
cat_summary = aggregate_length_metrics(df_lengths, "category")

print("global:")
print(global_summary)
print()

print("by category:")
print(cat_summary)
print()

global:
context_length          20.54
utterance_length         4.59
options_total_length    17.41
options_mean_length      4.35
item_total_length       42.54
dtype: float64

by category:
                                context_length                       \
                                          mean median   std min max   
category                                                              
baseline_focus_object                    15.35   15.0  1.62  13  20   
baseline_focus_subject                   15.24   15.0  2.01  13  20   
baseline_focus_verb                      16.77   17.0  0.55  16  18   
correction_focus_object                  23.34   24.0  1.86  20  27   
correction_focus_subject                 23.10   23.0  1.97  19  28   
correction_focus_verb                    22.14   22.0  1.26  16  25   
explicit_question_focus_object           21.96   22.0  1.81  19  26   
explicit_question_focus_subject          22.80   23.0  2.64  20  28   
explicit_question_focus_verb    

# III. Measuring Lemma Frequency

In [55]:
# TODO